In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch12-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "12-03-uncertainty.py"

# 用沒參與擬合的分子核對預測區間

首次執行會下載 ESOL。若要修改分析，請編輯下一格的腳本，再依序執行該格與最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors as D

# ESOL 水溶解度採 log10(mol/L)。原始文獻 DOI 10.1021/ci034243x。
_, (train, valid, test), _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter="scaffold",
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)

from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel

def describe(smiles):
    mols = [Chem.MolFromSmiles(s) for s in smiles]
    return np.array([[D.MolLogP(m), D.MolWt(m), D.TPSA(m)] for m in mols])

rng = np.random.default_rng(42)
chosen = np.sort(rng.choice(len(train), 32, replace=False))
scaler = StandardScaler()
X = scaler.fit_transform(describe(train.ids[chosen]))
X_valid = scaler.transform(describe(valid.ids))
kernel = ConstantKernel(1.0) * RBF(1.0) + WhiteKernel(0.1)
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, alpha=1e-8)
gp.fit(X, train.y.ravel()[chosen])
mu, sd = gp.predict(X_valid, return_std=True)
y = valid.y.ravel()
error = np.abs(y-mu)
inside = error <= 1.96 * sd
print("fitted kernel:", gp.kernel_)
print(f"95% coverage {inside.sum()}/{len(y)}, MAE {error.mean():.3f}")
order = np.argsort(sd, kind="stable")
for i in [order[0], order[-1]]:
    print(f"row {i}: {valid.ids[i]}, mean {mu[i]:.3f}, SD {sd[i]:.3f}, measured {y[i]:.3f}")
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
pos = np.arange(len(order))
ax.fill_between(pos, -1.96*sd[order], 1.96*sd[order], color="#d4e6e4",
                label="Model interval (95%)")
ax.scatter(pos, (y-mu)[order], color="#285561", s=14, label="Measured minus predicted")
ax.axhline(0, color="gray", lw=0.8)
ax.set(xlabel="Validation molecules, sorted by predicted SD", ylabel="Residual (log S units)")
ax.legend()
fig.savefig(out/"12-03-uncertainty.png", dpi=170)


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.removeprefix("This module requires PyTorch to be installed.\n").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "12-03-uncertainty.png")))